In [7]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# 031 — GCIM distributions, AvgSA([0, 3])

Computes the GCIM (Generalised Conditional Intensity Measure) target distributions for the
`(site, iml)` disaggregations that still need a record selection, conditioned on **AvgSA([0, 3])**.
For each site/intensity level the GMMs and correlation models are run over that site's
disaggregation to produce the target `stats` / `pdfs` / `cdfs`. This is the slow target-building
step that precedes record selection.

**Incremental caching** — the canonical record of "what has been selected and is still valid" is the
per-`(site, iml)` **stripe pickle + `.manifest.json`** written by nb `033`. This notebook calls
`find_stale_stripes` to compare the wanted key set (the per-site `union` lists) against those
manifests and computes gcim **only for the missing/stale stripes**, then **merges** them into
`gcim_dist_AvgSA_03.pickle` (so that file stays the complete gcim set — nothing already computed is
dropped). The per-stripe fingerprint (`stripe_input_fingerprint`) deliberately **excludes** the
IML-subset JSON, so appending IMLs to the `union` lists makes only the *new* `(site, iml)` stale —
every already-selected stripe is skipped here and in `032`/`033`. Set `FORCE_RECOMPUTE = True` to
treat all wanted stripes as stale.

**Upstream** — read by `setup_AvgSA03_gcim_gm_selection()` (`setup_AvgSA03_gm_selection.py`):

| Input | Config key |
|---|---|
| IML-based disaggregations (nb 021) | `cfg["proc_data"]["AvgSA_03_disagg_data_gm_selection"]` |
| Per-(site, imt, iml) poe stats | `cfg["proc_data"]["AvgSA_03_disagg_stats_gm_selection"]` |
| Per-site IML subset (one MSA stripe each) | `cfg["proc_data"]["AvgSA_03_imls_for_selection"]` |
| Site model | `cfg["hazard_models"]["eshm20_wp1_site_model"]` |
| Median-branch GMM logic tree | `cfg["hazard_models"]["eshm20_AvgSA_03_median_lt"]` |
| Correlation flatfiles + GM database | `cfg["proc_data"]["corr_model"]`, `cfg["proc_data"]["gm_database"]` (db used for selection, not the gcim calc) |

**Downstream** — `gcim_dist_AvgSA_03.pickle` (the complete `(site, iml)`-keyed gcim set, with only
the stale/new keys refreshed this run) is consumed by
`032-gm_selection_AvgSA_03_stage1_compute.ipynb`.

**Output** — `cfg["proc_data"]["gcim_dists"] / "gcim_dist_AvgSA_03.pickle"`.

**Run order** — run top to bottom, then `032` then `033`. On an all-valid run nothing is recomputed.

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pickagm.distributions import ensemble_ks_bounds

from phd_project.config.config import load_config 
from phd_project.scripts.cache_utils import fingerprint, load_or_compute
from phd_project.scripts.WP1_ground_motion_set.gm_selection import (
    calculate_gcim_distributions_for_sites,
    _selection_ctx_fingerprint_inputs,
    stripe_input_fingerprint,
    find_stale_stripes,
)

from phd_project.scripts.WP1_ground_motion_set.setup_AvgSA03_gm_selection import (
    setup_AvgSA03_gcim_gm_selection,
    SELECTION_CONFIG,
    stripe_source_fps,
)

cfg = load_config()

In [ ]:
# set up the gcim / record selection
site_iml_disaggs, disagg_stats, site_model, basic_selection_ctx, _ = setup_AvgSA03_gcim_gm_selection()
percentiles = SELECTION_CONFIG["percentiles"]

# Incremental cache: only (re)build gcim for stripes that are missing or stale on
# disk. A (site, iml) stripe is valid iff its result pickle + .manifest.json match
# its per-stripe fingerprint (stripe_input_fingerprint), which EXCLUDES the IML
# subset JSON. So appending IMLs to the "union" lists makes only the NEW (site, iml)
# stale; every already-selected stripe stays valid and is skipped.
RESULT_FOLDER = cfg["results"]["AvgSA_03_record_selection"]
source_fps_stripe = stripe_source_fps()
wanted = list(site_iml_disaggs.keys())
fp_fn = lambda s, i: stripe_input_fingerprint(
    s, i, source_fps_stripe, basic_selection_ctx, SELECTION_CONFIG)
to_compute, valid = find_stale_stripes(wanted, RESULT_FOLDER, fp_fn)
print(f"{len(wanted)} wanted (site, iml): {len(valid)} already valid, "
      f"{len(to_compute)} to (re)compute.")

In [ ]:
# Escape hatch: force-recompute gcim for ALL wanted stripes, ignoring the per-stripe
# manifests. Leave False for normal incremental runs (only stale/new stripes rebuild).
FORCE_RECOMPUTE = False
if FORCE_RECOMPUTE:
    to_compute = list(wanted)

gcim_dist_fp = cfg["proc_data"]["gcim_dists"] / "gcim_dist_AvgSA_03.pickle"

In [ ]:
# Compute the GCIM distributions ONLY for the stale/new (site, iml), then MERGE them
# into the existing gcim pickle so it stays the complete set — nothing already computed
# is dropped, only the stale/new keys are (re)written. 032 consumes it immediately.
batch = {k: site_iml_disaggs[k] for k in to_compute}
if batch:
    new_gcim = calculate_gcim_distributions_for_sites(
        batch, disagg_stats, site_model, basic_selection_ctx, percentiles)

    gcim_dists = {}
    if gcim_dist_fp.is_file():
        with open(gcim_dist_fp, "rb") as f:
            gcim_dists = pickle.load(f)
    gcim_dists.update(new_gcim)   # overwrite only the recomputed keys

    gcim_dist_fp.parent.mkdir(parents=True, exist_ok=True)
    with open(gcim_dist_fp, "wb") as f:
        pickle.dump(gcim_dists, f)
    print(f"(Re)computed gcim for {len(new_gcim)} (site, iml); "
          f"pickle now holds {len(gcim_dists)} total -> {gcim_dist_fp.name}")
else:
    # Nothing stale: load the existing complete gcim (if any) so later cells still work.
    if gcim_dist_fp.is_file():
        with open(gcim_dist_fp, "rb") as f:
            gcim_dists = pickle.load(f)
    else:
        gcim_dists = {}
    print("All stripes valid — no gcim to compute. Proceed to 032/033 to refresh CSVs if needed.")